**Linda Zier**

**ST 554**

**Final Project**

**Goal**

For this project we :

*   added our project and files to our github repo, committing often to show our progress.
*   wrote a Jupyter notebook that fits a machine learning model using pyspark’s MLlib module. In that same notebook we wrote code to read in a stream of data (data that we produced ourselves using a .py file that is also kept in the repo).
*   we used the model to do predictions on the stream and wrote those out to the console.


**Data**

The data is modified from the UCI machine learning repository. The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The study was about relating power consumption from different zones of Tetouan city to various factors like time of day, temperature, and
humidity.


*   We used a chunk to build our model.
*   We then 'streamed data' to a folder that we monitored. As data came in we used our fitted model to predict on the incoming data.





# Train Models

We created a Jupyter notebook for the model fitting part and the streaming part below. We completed the following:

*   read the data into a standard pandas data frame using the pd.read_csv() function
*   converted this to a spark data frame
*   treated the Power_Zone_3 variable as our response variable and used the other variables as predictors

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 11:43:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


**Creating the Pipeline**

We fit an elastic net model using CV with the steps below.
The transformations below each used an MLlib function that we put into a pipeline.

*   We used an SQL transformer to cast the hour variable as a DoubleType.

*   We binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).

*   The month column was one-hot encoded.
*   We Ran a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. We did this by:
    - first by using a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator
    
    - then we had a PCA transformer for use in our pipeline.
    - we used two PCs in our transformation.


*   We renamed our response variable as label

*   We used VectorAssembler() to put our predictors into a features. The predictors are:

    – two fitted PCA features

    – binary Hour variable

    – Power_Zone_1

    – Power_Zone_2

    – Month indicator variables

This completes our pipeline of transformations!


In [2]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("transformations complete")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

transformations complete


In [3]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, sql_label, assembler])

print("pipeline complete")

pipeline complete


Next we used the CrossValidator() function and the LinearRegression() function to fit an elastic net model. We did multiple combinations of reg and elastic net parameters.  We fit the model using 5-fold cross validation with root mean square error (RMSE) as the criteria: we're training 5 separate models (one per fold) and averaging their RMSE's together to get their RMSE for that combination. We then report the optimal values chosen for the tuning parameters and the CV error which is the RMSE from the best model.


In [4]:

# setting up elastic net model
lr= LinearRegression(elasticNetParam=0.5)

#  grid for the regParam and elasticNetParam
paramGrid= ParamGridBuilder() \
    .addGrid(lr.regParam,[0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam,[ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# 5-fold CV with rmse evaluator
cv=CrossValidator(estimator=lr,
                   estimatorParamMaps = paramGrid,  
                   evaluator = RegressionEvaluator(metricName= 'rmse'),
                   numFolds=5)

# fit the model
cvModel=cv.fit(transformedDF)

# Report the optimal values chosen for the tuning parameters
best = cvModel.bestModel
print(cvModel.bestModel.extractParamMap())

print("maybe do this instead")
print("Optimal regParam:", cvModel.bestModel.stages[-1].getRegParam())
print("Optimal elasticNetParam:", cvModel.bestModel.stages[-1].getElasticNetParam())

# report RMSE errors
print("RMSE errors= ", cvModel.avgMetrics)

# report lowest RMSE
print("CV RMSE = ", min(cvModel.avgMetrics))

# report training set RMSE
predictions = cvModel.transform(powerDF)
trainRMSE = RegressionEvaluator(metricName= 'rmse').evaluate(predictions)
print("Training RMSE:", trainRMSE)

IllegalArgumentException: [FIELD_NOT_FOUND] No such struct field `prediction` in `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, `Diffuse_Flows`, `Power_Zone_1`, `Power_Zone_2`, `Power_Zone_3`, `Month`, `Hour`, `CrossValidator_c890c6966959_rand`, `HourD`, `Hour_bin`, `Month_ohe`, `pca_input`, `pca_features`, `label`, `features`. SQLSTATE: 42704